# 09 — Local Plate Solve via solve-field

Validate the local `solve-field` backend in `extractor/platesolve.py`.

**Run this notebook from the `Python (spectrangle WSL)` kernel.**  
The `solve-field` binary is installed in Ubuntu/WSL; it is not available from a Windows kernel.

Remote solving still works unchanged — just use `backend='remote'`.

## Cell 1 — Environment check

In [1]:
import sys
import shutil
from pathlib import Path

# Add spectrangle/ to sys.path so 'import extractor' works without installing.
# Path('..') from notebooks/ resolves to spectrangle/ which contains extractor/.
_pkg_root = str(Path('..').resolve())
if _pkg_root not in sys.path:
    sys.path.insert(0, _pkg_root)

print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version}")

sf = shutil.which("solve-field")
print(f"\nsolve-field found : {sf is not None}")
if sf:
    print(f"solve-field path  : {sf}")
else:
    print("  *** solve-field NOT found — run from the WSL kernel or install astrometry.net ***")

# Edit these paths if your setup differs
BACKEND_CFG = Path("/mnt/c/Users/bassd/.astrometry/backend.cfg")
INDEX_DIR   = Path("/mnt/c/Users/bassd/astrometry-data/4100/")

print(f"\nBackend config    : {BACKEND_CFG}")
print(f"  exists          : {BACKEND_CFG.exists()}")

print(f"\nIndex directory   : {INDEX_DIR}")
print(f"  exists          : {INDEX_DIR.exists()}")
if INDEX_DIR.exists():
    idx_files = sorted(INDEX_DIR.glob("*.fits"))
    print(f"  index files     : {len(idx_files)}")
    if idx_files:
        print(f"  first           : {idx_files[0].name}")
        print(f"  last            : {idx_files[-1].name}")

try:
    import extractor
    print(f"\nextractor version : {extractor.__version__}")
    print(f"extractor path    : {extractor.__file__}")
except ImportError as e:
    print(f"\nextractor import error: {e}")

Python executable : /mnt/c/Users/bassd/Research/Spectra Angle/spectrangle/.venv/bin/python
Python version    : 3.11.0rc1 (main, Aug 12 2022, 10:02:14) [GCC 11.2.0]

solve-field found : True
solve-field path  : /usr/bin/solve-field

Backend config    : /mnt/c/Users/bassd/.astrometry/backend.cfg
  exists          : True

Index directory   : /mnt/c/Users/bassd/astrometry-data/4100
  exists          : True
  index files     : 5
  first           : index-4115.fits
  last            : index-4119.fits

extractor version : 0.2.0
extractor path    : /mnt/c/Users/bassd/Research/Spectra Angle/spectrangle/extractor/__init__.py


## Cell 2 — Load test image

In [2]:
import numpy as np
from astropy.io import fits

# Edit this path if your test image is elsewhere
IMAGE_PATH = Path("../data/fuji6_asi178_100_15s.fit")

with fits.open(IMAGE_PATH) as hdul:
    image           = hdul[0].data.astype(np.float32)
    original_header = hdul[0].header.copy()

IMAGE_HEIGHT, IMAGE_WIDTH = image.shape[-2], image.shape[-1]

print(f"Image file  : {IMAGE_PATH.name}")
print(f"Dimensions  : {IMAGE_WIDTH} x {IMAGE_HEIGHT} px")
print(f"dtype       : {image.dtype}")
print(f"Value range : {image.min():.0f} – {image.max():.0f}")

Image file  : fuji6_asi178_100_15s.fit
Dimensions  : 3096 x 2080 px
dtype       : float32
Value range : 0 – 1


## Cell 3 — Source detection / xylist

In [3]:
from extractor import extract_stars, make_xylist

MAX_SOURCES = 300

xs, ys, fluxes = extract_stars(
    image,
    max_sources=MAX_SOURCES,
    mask_spectra=True,
)

print(f"Detected sources : {len(xs)}")
print(f"Flux range       : {fluxes.min():.0f} – {fluxes.max():.0f}")

xylist_buf = make_xylist(xs, ys)
print(f"xylist buffer    : {len(xylist_buf.getvalue())} bytes")

Detected sources : 255
Flux range       : 0 – 15
xylist buffer    : 11520 bytes


## Cell 4 — Run local plate solve

Edit `TWEAK_ORDER`, `SCALE_LOW`, `SCALE_HIGH`, or `OUTPUT_DIR` as needed.

In [4]:
from extractor.platesolve import solve_plate, PlateSolveError, LocalSolveFieldError

# ---- parameters (edit freely) ----
TWEAK_ORDER = 5
SCALE_LOW   = 60.0
SCALE_HIGH  = 95.0
OUTPUT_DIR  = Path("../working/local_solve_09")
# ----------------------------------

try:
    result = solve_plate(
        xs=xs, ys=ys,
        image_width=IMAGE_WIDTH,
        image_height=IMAGE_HEIGHT,
        original_header=original_header,
        backend="local",
        backend_config=str(BACKEND_CFG),
        scale_units="arcsecperpix",
        scale_low=SCALE_LOW,
        scale_high=SCALE_HIGH,
        tweak_order=TWEAK_ORDER,
        output_dir=OUTPUT_DIR,
        timeout=300,
        verbose=True,
    )
except LocalSolveFieldError as e:
    print(f"\nLocalSolveFieldError:\n{e}")
    result = None

if result is not None:
    print("\n=== Solve succeeded ===")
    print(f"Status          : {result.status}")
    wcs_path = OUTPUT_DIR / "xylist.wcs"
    print(f".wcs file       : {wcs_path} (exists={wcs_path.exists()})")
    if result.corr_table is not None:
        print(f"Matched sources : {len(result.corr_table)}")
    for prod, status in result.fetch_status.items():
        print(f"  {prod:<8}: {status}")
else:
    print("\n=== No result returned ===")

Wrote 255 sources → ../working/local_solve_09/xylist.fits
Running: /usr/bin/solve-field ../working/local_solve_09/xylist.fits --width 3096 --height 2080 --scale-units arcsecperpix --scale-low 60.0 --scale-high 95.0 --tweak-order 5 --dir ../working/local_solve_09 --backend-config /mnt/c/Users/bassd/.astrometry/backend.cfg --overwrite --no-plots
  Reading input file 1 of 1: "../working/local_solve_09/xylist.fits"...
  Solving...
  Reading file "../working/local_solve_09/xylist.axy"...
  Field 1 did not solve (index index-4119.fits, field objects 1-10).
  Field 1 did not solve (index index-4118.fits, field objects 1-10).
    log-odds ratio 60.2687 (1.49402e+26), 21 match, 0 conflict, 40 distractors, 179 index.
    RA,Dec = (113.419,30.4343), pixel scale 80.6262 arcsec/pix.
    Hit/miss:   Hit/miss: -----+-++--+++---++--------++++--++++-------------+----+++--+(best)-------++-------+---++----+---+--------
  Field 1: solved with index index-4117.fits.
  Field 1 solved: writing to file ../wor

## Cell 5 — WCS center metrics

In [5]:
from astropy.wcs import WCS
from extractor.wcsangle import center_wcs_angle_metrics
from extractor.platesolve import wcs_summary

assert result is not None, "No result — re-run Cell 4 first."

wcs     = WCS(result.header)
metrics = center_wcs_angle_metrics(wcs, image.shape, compute_east=True)

print("WCS summary:")
print(wcs_summary(result.header))

print(f"\nCenter pixel      : ({metrics.x_center:.1f}, {metrics.y_center:.1f})")
print(f"Center RA / Dec   : {metrics.ra_deg:.6f}°  /  {metrics.dec_deg:.6f}°")
print(f"North angle       : {metrics.north_angle_deg:.4f}°")
if metrics.east_angle_deg is not None:
    print(f"East angle        : {metrics.east_angle_deg:.4f}°")

WCS summary:
  CTYPE1  : RA---TAN-SIP
  CTYPE2  : DEC--TAN-SIP
  CRVAL1  : 120.62911
  CRVAL2  : 29.793897
  CRPIX1  : 1286.8585
  CRPIX2  : 1140.5638
  CD1_1   : -0.0217091
  CD1_2   : 0.00518993
  CD2_1   : -0.00515521
  CD2_2   : -0.021738
  SIP     : yes (order 5)

Center pixel      : (1547.5, 1039.5)
Center RA / Dec   : 113.423590°  /  30.426436°
North angle       : -107.0058°
East angle        : 163.0386°


Set MJD-AVG to 61140.040531 from DATE-AVG.
Set MJD-END to 61140.040618 from DATE-END'. [astropy.wcs.wcs]


## Cell 6 — Parameter sweep

Edit `TEST_CONFIGS` and re-run to compare solve outcomes across parameter combinations.
By default only the two most interesting tweak orders are active; uncomment others.

In [6]:
TEST_CONFIGS = [
    dict(tweak_order=5, scale_low=70.0, scale_high=95.0),
    dict(tweak_order=3, scale_low=70.0, scale_high=95.0),
    # dict(tweak_order=2, scale_low=70.0, scale_high=95.0),
    # dict(tweak_order=4, scale_low=70.0, scale_high=95.0),
    # dict(tweak_order=5, scale_low=65.0, scale_high=100.0),  # wider scale window
]

sweep_results = {}

for cfg in TEST_CONFIGS:
    label   = f"tweak{cfg['tweak_order']}_s{cfg['scale_low']:.0f}-{cfg['scale_high']:.0f}"
    out_dir = Path(f"../working/local_solve_09_{label}")
    print(f"\n--- {label} ---")
    try:
        r = solve_plate(
            xs=xs, ys=ys,
            image_width=IMAGE_WIDTH, image_height=IMAGE_HEIGHT,
            original_header=original_header,
            backend="local",
            backend_config=str(BACKEND_CFG),
            scale_units="arcsecperpix",
            output_dir=out_dir,
            timeout=300,
            verbose=False,
            **cfg,
        )
        if r is not None:
            m = center_wcs_angle_metrics(WCS(r.header), image.shape, compute_east=False)
            n_matched = len(r.corr_table) if r.corr_table is not None else "?"
            sweep_results[label] = dict(
                success=True,
                north_angle=m.north_angle_deg,
                ra=m.ra_deg, dec=m.dec_deg,
                n_matched=n_matched,
            )
            print(f"  OK | matched={n_matched} | north={m.north_angle_deg:.4f}°  "
                  f"RA={m.ra_deg:.4f}°  Dec={m.dec_deg:.4f}°")
        else:
            sweep_results[label] = dict(success=False, reason="no result")
            print("  FAILED (no result / no solution)")
    except LocalSolveFieldError as e:
        sweep_results[label] = dict(success=False, reason=str(e)[:120])
        print(f"  LocalSolveFieldError: {e!s:.120}")

print("\nSweep complete.")


--- tweak5_s70-95 ---


Set MJD-AVG to 61140.040531 from DATE-AVG.
Set MJD-END to 61140.040618 from DATE-END'. [astropy.wcs.wcs]


  OK | matched=86 | north=-107.0058°  RA=113.4236°  Dec=30.4264°

--- tweak3_s70-95 ---
  OK | matched=51 | north=-106.7971°  RA=113.4008°  Dec=30.4010°

Sweep complete.


Set MJD-AVG to 61140.040531 from DATE-AVG.
Set MJD-END to 61140.040618 from DATE-END'. [astropy.wcs.wcs]


## Cell 7 — Summary

In [7]:
print("=" * 62)
print("  LOCAL PLATE SOLVE SUMMARY")
print("=" * 62)
print(f"  Backend        : local (solve-field at {sf or 'NOT FOUND'})")
print(f"  Backend config : {BACKEND_CFG}")
print(f"  Tweak order    : {TWEAK_ORDER}")
print(f"  Scale range    : {SCALE_LOW}–{SCALE_HIGH} arcsec/px")
print(f"  Sources sent   : {len(xs)}")

if result is not None:
    n_match = len(result.corr_table) if result.corr_table is not None else "?"
    print(f"  Sources matched: {n_match}")
    print(f"  Solve status   : {result.status}")
    print(f"  Center RA/Dec  : {metrics.ra_deg:.6f}°  /  {metrics.dec_deg:.6f}°")
    print(f"  North angle    : {metrics.north_angle_deg:.4f}°")
    if metrics.east_angle_deg is not None:
        print(f"  East angle     : {metrics.east_angle_deg:.4f}°")
else:
    print("  Solve status   : FAILED")

if sweep_results:
    print()
    print("  Parameter sweep:")
    for label, info in sweep_results.items():
        if info.get("success"):
            print(f"    {label:<40}  north={info['north_angle']:.4f}°  matched={info['n_matched']}")
        else:
            print(f"    {label:<40}  FAILED  ({info.get('reason', ''):.60})") 

print("=" * 62)

  LOCAL PLATE SOLVE SUMMARY
  Backend        : local (solve-field at /usr/bin/solve-field)
  Backend config : /mnt/c/Users/bassd/.astrometry/backend.cfg
  Tweak order    : 5
  Scale range    : 60.0–95.0 arcsec/px
  Sources sent   : 255
  Sources matched: 86
  Solve status   : success
  Center RA/Dec  : 113.423590°  /  30.426436°
  North angle    : -107.0058°
  East angle     : 163.0386°

  Parameter sweep:
    tweak5_s70-95                             north=-107.0058°  matched=86
    tweak3_s70-95                             north=-106.7971°  matched=51
